In [ ]:
!pip install transformers datasets torchaudio librosa soundfile --quiet


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 64.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 35.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 43.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 13.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 87.1 MB/s eta 0:00:00


In [ ]:
from google.colab import files
uploaded = files.upload()  # Upload a .wav file

Saving Recording.wav to Recording.wav


In [ ]:
import torchaudio
file_path = list(uploaded.keys())[0]
waveform, sample_rate = torchaudio.load(file_path)

In [ ]:
from transformers import Wav2Vec2Processor, Wav2Vec2ForCTC
import torch
import librosa

# Load processor and model
processor = Wav2Vec2Processor.from_pretrained("facebook/wav2vec2-base-960h")
model = Wav2Vec2ForCTC.from_pretrained("facebook/wav2vec2-base-960h")

# Resample if needed (e.g., 48000 Hz → 16000 Hz)
waveform_resampled = librosa.resample(waveform.numpy()[0], orig_sr=sample_rate, target_sr=16000)


Some weights of Wav2Vec2ForCTC were not initialized from the model checkpoint at facebook/wav2vec2-base-960h and are newly initialized: ['wav2vec2.masked_spec_embed']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [ ]:
# Prepare input
inputs = processor(waveform_resampled, sampling_rate=16000, return_tensors="pt", padding=True)

# Predict
with torch.no_grad():
    logits = model(inputs.input_values).logits

# Get predicted IDs
predicted_ids = torch.argmax(logits, dim=-1)

# Decode to text
transcription = processor.batch_decode(predicted_ids)
print("Predicted Transcription:", transcription[0])


Predicted Transcription: THIS IS A TESTAPPLICATION WELCOME EVERYBODY LET'S LEARN SPEECH RECOGNITION THANK YOU
